- RNN의 한계
    - Attention의 등장 배경 및 장점
    - Attention is All You Need
- Self-Attention
    - Query, Key, Value 개념
    - Cross Attention vs. Self-Attention 개념 및 과정
    - Scaled Dot-Product Attention
- Multi-Head Attention
    - Multi-Head Attention 개념
    - Multi-Head Attention 작동 방식
- Transformer 전체 아키텍처
    - 전처리 단계(토큰화 - 임베딩 - Positional Encoding)
    - Encoder와 Decoder

# 1. RNN의 한계
- 장기 의존성 문제
- 기울기 소실 문제: 데이터가 길어질수록, 데이터의 후반분에서 초반에 위치한 정보가 잘 반영이 되지 않음

# 2. Attention
**등장배경**
- Seq2Seq의 병목 현상: 고정된 Context Vector에 모든 정보를 압축하려다 보니 **중요한 정보가 손실**, 디코더가 **매번 같은 Context Vector만 참고**하여 출력 단어마다 필요한 정볼르 다르게 반영하기 어려움.

- Attention 개념: 결과를 만들 때 입력값의 모든 부분을 똑같이 참고하는 것이 아니라, **현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여하여 활용**
- Seq2Seq와 차이점: 출력시점마다 새로운 context vector를 동적으로 계산함 -> 병렬화
- 처리 과정:
	- 1. 입력을 벡터 형태로 변환: 입력 문장을 숫자 벡터로 변환
	- 2. 단어별 중요도 계산: 어떤 단어에 집중을 해야 할 지 산출
	- 3. 중요도에 따른 입력 정보 조합
- 장점:
	1. 시간 경과에 따른 유연성: 모든 단어에서 다른 모든 단어로 직접 연결될 수 있어서 장기 의존성 문제 완화
	2. 공간에 대한 유연성: CNN->이미지의 지역적인 부분을 중심으로 정보 처리 vs Attention: 전체 이미지의 전역적인 관계 학습
	3. **병렬화**: self-attention에서 문장 내 모든 단어의 관계를 한번에, 독립적으로 계산 -> 병렬처리 가능, 긴 시퀀스도 더 효율적으로 학습.

# 2.2 Attention is All you need
- Google brain의 논문
- Transformer 구조를 처음 발표 (Self-Attention 방식)
=> Attention을 쓰던 딥러닝 모델들이 대부분 Self-Attention 방식을 채택
- 주요 내용
	- 논문배경: RNN과 RNN기반 모델들의 한계점 -> 오직 Attention만을 사용하는 모델을 만들면 어떨까?
	- Transformer 등장: RNN 순환 구조를 완전히 제거하고, self-attention에 기반한 Transformer 모델 제안
	- 중요한 발견: 성공적인 모델링에 Recurrence 및 Convolution이 필수가 아님을 증명 -> Attention만으로 더 좋은 성능을 낼 수 있음 <br> +) 모든 단어들을 동시에 계산할 수 있는 병렬화 가능.
	- 파급효과: LLM 탄생, NLP 평정, 컴퓨터 비적, 음성 처리, 신약 개발 등 타분야로의 확장

# 2.3 Transformer
- 개념: RNN의 순차적인 계산 방식을 완전히 버리고, 오직 어텐션(Attention)만으로 문장의 의미와 구조를 파악하는 모델
- 기존 모델과 차이점: RNN 기반 Encoder/Decoder+Attention -> Attention 기반 Encoder/Decoder
- 핵심 기술:
	- Self-Attention: 문장 안에서 어떤 단어가 다른 단어들과 얼마나 중요한 관계를 맺고 있는지 한 번에 파악
	- Multi-Head Attention: 셀프 어텐션을 여러 개의 머리(Head)로 동시에, 서로 다른 관점에서 실행
	- Positional Encoding: 단어의 위치 정보를 벡터에 추가하여 각 단어가 문장의 몇 번째 위치에 존재하는지 알 수 있음.

# 3. Self-Attention
- Attention의 기본 구조: Query, Key, Value

- 쿼리(Query): **'질문'** 또는 **'요청'**으로, 지금 당장 내가 알고 싶거나, 초점을 맞추고 있는 대상
	- ex) "'The cat is sleeping'에서 cat이 뭐지?" 가 Query
- 키(Key): **모든 정보들이 달고 있는 '이름표' 또는 '색인' **
	- ex) 'The -> 관사, Cat -> 동물정보, is -> 동사, sleeping -> 수면정보' 이게 key
- 벨류(value): 키와 한 쌍으로 묶여 있는 **'실제 내용물'**. <br>쿼리와 키의 관련도 계산이 끝나고, 가장 관련성이 높다고 판단된 키가 선택되면, 모델은 그 키에 해당하는 밸류를 가져와 사용.
	- ex) Key가: "나는 잠자는 행동 관련" 이라고 소개하면, Value는: "잠자는 의미의 실제 정보" 를 담고 있음.

## 3.1 Cross-Attention vs Self Attention
- Cross-Attention 개념: 다음 단어를 만들 때 입력 문장 중 중요한 단어에 집중하는 방법
- 특징: 하나의 시퀀스가 완전히 다른 시퀀스를 참고하여 정보를 만들어냄
  - Query와  Key - Value쌍이 서로 다른 시퀀스에서 출력
- 핵심과정:
1. 먼저 Encoder는 입력 문장의 **각 단어를 벡터 형태로 변환**하여 저장한다. 이때 **각 단어는 Key와 Value** 정보를 가진다.

2. Decoder는 현재까지 생성한 단어들을 바탕으로 “<u>다음 단어를 만들기 위해 어떤 정보를 참고해야 하는가?”를 나타내는 Query</u>를 만든다.

3. 이후 Query와 Encoder의 각 Key를 비교하여 **관련성(유사도)을 계산**한다. <u>관련성이 높을수록 해당 단어가 현재 번역에 더 중요</u>하다는 의미이다.

4. 계산된 점수는 **softmax를 통해 가중치로 변환**되며, Decoder는 이 가중치를 사용해 **각 Value를 중요도에 따라 합친다.** 이렇게 만들어진 벡터를 **Context Vector**라고 한다.

5. 마지막으로 Decoder는 자신의 현재 상태와 Context Vector를 함께 사용하여 다음 단어를 예측한다.

**반면에 Self Attention은 하나의 시퀀스에서 Query와  Key - Value쌍이 출력!**

ex) The cat is sleeping -> 여기서 sleeping은:“누가 자고 있지?”를 알기 위해 cat을 참고함.

즉 입력문장 안의 단어들이 스스로 Q/K/V를 만들고 서로 참고한다는 뜻

- 핵심 과정:
1. 단어의 프로필 만들기(벡터 임베딩)
2. 단어 간 관계 점수 계산 (내정 및 정렬 점수)
3. 중요도 배분 (소프트맥스 & 어텐션 가중치)
4. 새로운 벡터 생성

# 4. Scaled-Dot Product Attention
## 4.1 Dot-Product Attention
- 개념: Attention Score를 계산하는 기본적인 방법

-> 쿼리와 벡터를 내적하여 유사도를 구함.
- 문제점: 벡터 차원이 커질수록 내적값이 지나치게 커지거나 작아지는 **Attention Score의 극단화**

->이런 극단적인 값이 softmax에 입력되면 특정 단어에 확률이 과도하게 집중되고, gradient가 0에 가까워지는 **기울기 소실 문제**가 발생 가능.

-> 그래서 등장한 것이 Scaled Dot-Product Attention!

## 4.2 Scaled Dot-Product Attention

-개념: Transformer의 "어텐션 계산 공식" 자체를 의미!
- 해결 방법: Dot-Production Attention 과정에서 내적을 진행한 후, 값의 크기를 맞추는 **스케일링** 과정을 추가

- 유사도 계산 후, 계산된 값을 **√dₖ로 나누어 스케일링**
- Scaled Dot-Product Attention 계산의 결과로 얻어지는 벡터:
  - Cross-Attention: 현재 생성 중인 단어가, 입력 문장에서 어떤 단어들에 집중해야 하는지를 반영한 **Context Vector**
  - Self-Attention: 문장 내 다른 단어들과의 관계를 반영한 **문맥적 표현**


# 5. Multi-Head Attention
- 개념: 셀프 어텐션을 **여러 개의 머리(head)로 동시에, 서로 다른 관점에서 실행함**
- 등장배경: Single-Head Attention의 한계
  - 작동 방식: 문장 내 단어들 간의 관계를 파악하기 위해 하나의 가중치 행렬만을 학습하고 번역에 사용.
  - 512차원의 벡터를 입력하고 이 벡터를 위한 어텐션 가중치 분포를 계산 -> 가중합하여 새로운 512차원 벡터 출력
  - 즉, **단어 관계를 보는 눈이 한개 뿐**
  - 결과: "The animal didn't cross the stree because **it** was too tired" 라는 단어에서, it은 여러단어와 관계가 있지만, **하나의 어텐션 점수**로 섞여버려 단어 간 중요한 관계를 놓침

- **Multi-head Attention:** Attention을 여러 개 사용
- 한 단어와 다른 단어 간의 관계를 여러 차원으로 나누어 **병렬로 학습**

ex)
Head 1 → 문법 관계 집중

Head 2 → 의미 관계 집중

Head 3 → 위치 관계 집중

- 작동 방식
  1. **분할** - 512차원 벡터를 **8개의 서로 다른 관점으로 나눔** => 64차원 벡터들로 분할
  2. **병렬 어텐션 계산**- 8개의 헤드는 서로에게 전혀 간섭하지 않고, 병렬로 Scaled Dot-Product Attention을 계산 -> 각 헤드의 관점에서 문맥을 이해한 64차원의 결과 벡터 도출
  3. **결합 및 최종 투영** - 각 8개의 헤드가 내놓은 8개의 분석 결과를 하나로 합침



# 6. Transformer 전체 아키텍처
## 6.1 트랜스포머의 전처리 단계
1. 토큰화 - 입력 텍스트를 모델이 처리할 수 있는 **단위**로 나누는 첫 번째 단계
  - 대부분의 경우 각 토큰은 하나의 **단어**에 해당

2. 임베딩 - 토큰화된 각 단위(토큰)은 임베딩 단계에서 **숫자의 벡터**로 변환
  - 인간의 언어를 컴퓨터의 언어로 번역하는 다리 역할
  - 원리: 단어의 의미를 **공간적으로 표현**
  -> 유사한 단어는 유사한 좌표로 변환

3. Positional Encoding - 단어들의 **순서 정보**를 임베딩 벡터에 추가하는 단계
  - Transformer는 단어 순서를 알지 못하기 때문에 문맥을 파악하기 위해 순서 정보(위치 정보) 제공해야함

  - 동작 방식: 문장에서 차지하는 위치가 다르면, 다른 좌표를 가지게 됨

## 6.2 트랜스포머의 Encoder와 Decoder

### 6.2.1 Encoder
- 개념: 입력 문장을 이해하고 요약된 의미 벡터로 변환하는 역할
- 구조: 전처리 과정 및 **인코더 레이어(Multi-Head Attention, Feed Forward Layer)** 로 구성되어 있음

- Feed-Forward란?
  - 입력 벡터의 차원을 확장하고 비선형 변환(ex: ReLU)을 적용하여 새로운 표현을 생성하는 신경망 구조
  - Attention에서 어떤 단어를 더 집중해서 볼지를 정했다면, Feed-Forward에서는 **그 단어에 대해 심층 분석**을 하는 것

- Encoder Layer 처리 구조
  
    Input

    ↓

  Self-Attention
  
    ↓

  Add (Input + Attention 결과)  ← 잔차 연결

    ↓

  Feed Forward

  ↓

  Add (위 결과 + FFN 결과)     ← 또 잔차 연결

  ↓

  Output

- Residual connections(잔차 연결) 이란?
  - 입력값을 그대로 출력에 더해주는 것
  - 핵 이유: Attention이 뭔갈 변형을 함 -> 그냥 Attention 결과 쓰면 원래 정보가 완전히 바뀔 위험 있음
  - 기울기 소실 및 폭주 현상 방지

# 6.2.2 Decoder
- 개념:  Encoder가 분석한 입력 문장의 의미 벡터를 받아서 출력 문장을 순차적으로 생성하는 역할을 담당
- 구조: 전처리 과정, 디코더 레이어, 선형 레이어와 소프트맥스 레이어로 구성

- 디코더 레이어 구조:
1. **Masked Multi-Head Attention**
2. **Encoder-Decoder** Multi-Head **Attention (Cross-Attention)**
3. Feed-Forward Layer

- **Masked Multi-Head Attention**: 미래 시점의 단어 정보를 참고하지 못하도록 **마스크**를 적용하는 어텐션 메커니즘

-> 트랜스포머 디코더는 모델이 이미 생성한 단어들**(과거 정보)**만을 기반으로 다음 단어를 예측하도록 강제

- 작동 방식: 이를 위해 Look-Ahead Mask(Causal Mask)를 사용하여 현재 단어보다 오른쪽(미래)에 있는 토큰의 어텐션 값을 0으로 만들어 무시

- **Encoder-Decoder Attention (Cross-Attention)** : 디코더가 인코더의 출력(Context)을 참고하면서 현재 생성 중인 단어를 입력 문장의 의미와 연결하는 과정

- 효과: 디코더가 입력 문장 중 어떤 부분에 주목해야 하는지를 학습

# 전체 흐름 정리

1. 입력 준비: 문장을 [의미 + 위치] 벡터로 바꾸기

2. 인코더: 입력 문장의 [문맥적 의미] 깊이 이해하기

3. 디코더: 번역 문장을 [한 단어씩] 생성하기
